In [1]:
import warnings
warnings.filterwarnings('ignore')

%matplotlib qt
import numpy as np
import glob
import sys
import pickle as pkl
sys.path.append("..")

import mne
from scipy.io import loadmat

import socket
import pickle as pkl
import nibabel as nib
import json
import nibabel as nib

%load_ext autoreload
%autoreload 2

In [2]:
# %autoreload now
# import inspect
# print(inspect.signature(plot_cluster_bubble_heatmap))

# import helpers.source_level_psd_helpers as slph
# import inspect
# print(slph.__file__)
# print(inspect.signature(slph.plot_cluster_bubble_heatmap))
# print(inspect.signature(plot_cluster_bubble_heatmap))   # the one your calls actually use

In [3]:
device_name = socket.gethostname()
print(device_name)

DESKTOP-I0FNO1F


In [4]:
if device_name == "LAPTOP-LRI6535I":
    path_analysis_data = "E:/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    plots_path = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/analysis_plots/"

if device_name == "kritrim":
    path_analysis_data = "C:/Users/manda/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"

if device_name == "DESKTOP-I0FNO1F":
    path_analysis_data = "D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    # plots_path = "D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/"
    plots_path = "D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/"



In [5]:
from helpers.helper_functions import get_matched_legths, plot_significance_single_sub_band, get_fooof_aperiodic, build_vector_grid_pdf, plot_relative_difference_single_sub_band, get_fooof_bands, split_half_latex_table
from helpers.topo_eeg import get_fdr_p_values,  get_band_power_difference_all_subjects_significance_aperiodic_fdr, get_channel_wise_fooof
from helpers.source_level_psd_helpers import *
from helpers.group_level_helpers import group_level_bootstrap_test, latex_group_vs_individual_table
from helpers.frequency_analysis import compute_psd_mne


def store_mne_raw(eeg_filtered, path_folder, path_data, ovwrite=False):
    eeg_filtered.save(path_folder + path_data, overwrite=ovwrite)

def load_mne_raw(path_folder, path_data):
    return mne.io.read_raw_fif(path_folder + path_data, preload=True)



Python Executable Path: 
c:\Users\specsdcn\anaconda3\envs\pdoc\python.exe


d:\Taran\OneDrive - Radboud Universiteit\Documents\music_doc_study\analysis\..\helpers\helper_functions.py:9: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  import fooof


In [6]:
import warnings
warnings.filterwarnings('ignore')

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}

pat_state_dict = {
    'sub01': 'UWS',
    'sub03': 'MCS+',
    'sub04': 'MCS-',
    'sub07': 'MCS-',
    'sub09': 'MCS-',
    'sub10': 'MCS-',
    'sub12': 'MCS-',
}

sfreq = 512
paradigms = {0: "noise", 1: "music"}
pat_state = {"sub01": "UWS", "sub03": "MCS+", "sub04": "MCS-", "sub07":"MCS-", "sub09":"MCS-", "sub10":"MCS-", "sub12":"MCS-"}


#### Load AAL Atlas

In [7]:
aal_dir = path_analysis_data_new + "eeg_source/"

with open(f'{aal_dir}AAL_labels.json', 'r') as f:
    aal_labels = json.load(f)

In [8]:
aal_labels

['Precentral_L',
 'Precentral_R',
 'Frontal_Sup_L',
 'Frontal_Sup_R',
 'Frontal_Sup_Orb_L',
 'Frontal_Sup_Orb_R',
 'Frontal_Mid_L',
 'Frontal_Mid_R',
 'Frontal_Mid_Orb_L',
 'Frontal_Mid_Orb_R',
 'Frontal_Inf_Oper_L',
 'Frontal_Inf_Oper_R',
 'Frontal_Inf_Tri_L',
 'Frontal_Inf_Tri_R',
 'Frontal_Inf_Orb_L',
 'Frontal_Inf_Orb_R',
 'Rolandic_Oper_L',
 'Rolandic_Oper_R',
 'Supp_Motor_Area_L',
 'Supp_Motor_Area_R',
 'Olfactory_L',
 'Olfactory_R',
 'Frontal_Sup_Medial_L',
 'Frontal_Sup_Medial_R',
 'Frontal_Med_Orb_L',
 'Frontal_Med_Orb_R',
 'Rectus_L',
 'Rectus_R',
 'Insula_L',
 'Insula_R',
 'Cingulum_Ant_L',
 'Cingulum_Ant_R',
 'Cingulum_Mid_L',
 'Cingulum_Mid_R',
 'Cingulum_Post_L',
 'Cingulum_Post_R',
 'Hippocampus_L',
 'Hippocampus_R',
 'ParaHippocampal_L',
 'ParaHippocampal_R',
 'Amygdala_L',
 'Amygdala_R',
 'Calcarine_L',
 'Calcarine_R',
 'Cuneus_L',
 'Cuneus_R',
 'Lingual_L',
 'Lingual_R',
 'Occipital_Sup_L',
 'Occipital_Sup_R',
 'Occipital_Mid_L',
 'Occipital_Mid_R',
 'Occipital_Inf_L'

#### Load Source Reconstructed Data

In [9]:
# with open(path_analysis_data_new + "all_eeg_source_reconstruction_eloreta_45_adhoccov.pkl", "rb") as f:
#     eeg_source_reconstructed = pkl.load(f)


In [10]:
# eeg_source_reconstructed["sub01"]["rest_init"].shape

In [11]:
# eeg_source_reconstructed_cont = {}

# for sub in eeg_source_reconstructed:
#     eeg_source_reconstructed_cont[sub] = {}
#     for parad in  eeg_source_reconstructed[sub]:
#         eeg_source_reconstructed_cont[sub][parad] = eeg_source_reconstructed[sub][parad].transpose(1, 0, 2).reshape(116, -1)


In [12]:
# a = np.empty_like(eeg_source_reconstructed_cont[sub][parad])
# a.shape

In [13]:
# count_cerebrum_labels = 0 

# for label in aal_labels:

#     if label.startswith("Cerebelum") or label.startswith("Vermis"):
#         continue

#     else:
#         count_cerebrum_labels += 1

# count_cerebrum_labels

In [14]:
# eeg_source_reconstructed_cont_cerebrum = {}
# eeg_source_reconstructed_cont_cerebrum_raw = {}

# regions_all = eeg_source_reconstructed_cont[sub][parad].shape[0]

# n_regions = 90
# ch_types = ['eeg'] * n_regions

# info = mne.create_info(ch_names=ch_types, sfreq=512, ch_types=ch_types)

# for sub in eeg_source_reconstructed_cont:
#     eeg_source_reconstructed_cont_cerebrum[sub] = {}
#     eeg_source_reconstructed_cont_cerebrum_raw[sub] = {}

#     for parad in  eeg_source_reconstructed_cont[sub]:
#         eeg_source_reconstructed_cont_cerebrum[sub][parad] = np.zeros([count_cerebrum_labels, eeg_source_reconstructed_cont[sub][parad].shape[1]])

#         for reg in range(regions_all):
#             reg_label = aal_labels[reg]

#             if reg_label.startswith("Cerebelum") or reg_label.startswith("Vermis"):
#                 continue

#             else:
#                 eeg_source_reconstructed_cont_cerebrum[sub][parad][reg, :] = eeg_source_reconstructed_cont[sub][parad][reg, :]

#         eeg_source_reconstructed_cont_cerebrum_raw[sub][parad] = mne.io.RawArray(eeg_source_reconstructed_cont_cerebrum[sub][parad], info, verbose=False)

In [15]:
# eeg_source_reconstructed_cont_cerebrum_raw['sub07'].keys()

### Load Source Reconstructed PSDs

In [16]:
with open(path_analysis_data_new + "all_eeg_source_psd_eloreta_vector_nofilt.pkl", "rb") as f:
    eeg_source_psd = pkl.load(f)

In [17]:
eeg_source_psd["sub01"]["rest_init"][0].shape   # expect (116, 226, n_epochs)

(116, 223, 80)

In [18]:
n_cerebrum = 90   # AAL regions 1-90 are cerebral; 91-116 are cerebellum/vermis
assert not any(l.startswith(("Cerebelum", "Vermis")) for l in aal_labels[:n_cerebrum])

psds_all = {
    sub: {parad: [eeg_source_psd[sub][parad][0][:n_cerebrum],   # (90, n_freq, n_epochs)
                  eeg_source_psd[sub][parad][1]]                # freqs
          for parad in eeg_source_psd[sub]}
    for sub in eeg_source_psd
}

In [19]:
psd, freqs = psds_all["sub09"]["rest_init"]
plt.semilogy(freqs, psd.mean(axis=(0, 2)))
plt.axvspan(8, 12, alpha=0.15, color="orange")
plt.xlabel("Frequency (Hz)"); plt.title("sub09 rest, mean over 90 regions")
plt.show()

FOOOF

In [20]:
# band_powers_all = {sub: {parad: [] for parad in eeg_source_reconstructed_cont_cerebrum[sub]} for sub in eeg_source_reconstructed_cont_cerebrum}
# psds_all = {sub: {parad: [] for parad in eeg_source_reconstructed_cont_cerebrum[sub]} for sub in eeg_source_reconstructed_cont_cerebrum}

# subjects = list(eeg_source_reconstructed_cont_cerebrum.keys())
# paradigms = list(eeg_source_reconstructed_cont_cerebrum[subjects[0]].keys())


# f_min = bands["delta"][0]
# f_max = bands["gamma low"][1]

# segment_parads = ['pink_segs', 'music_segs']
# power_type = "relative"

# for sub in eeg_source_reconstructed_cont_cerebrum:
#     for parad in  eeg_source_reconstructed_cont_cerebrum[sub]:

#         # if parad in segment_parads:
#         #     segment_secs = 3

#         # else:
#         segment_secs = 5
        
#         segment_size = int(sfreq * segment_secs)
#         overlap = int(segment_size /2)

#         eeg_array = eeg_source_reconstructed_cont_cerebrum[sub][parad]
#         n_regions = eeg_array.shape[0]
#         ch_types = ['eeg'] * n_regions

#         info = mne.create_info(ch_names=ch_types, sfreq=sfreq, ch_types=ch_types)
#         eeg_raw = mne.io.RawArray(eeg_array, info, verbose=False)

#         band_powers, psds, freqs = compute_psd_mne(eeg_raw, bands, overlap, segment_size, f_min, f_max, power_type=power_type)

#         band_powers_all[sub][parad] = band_powers
#         psds_all[sub][parad] = [psds, freqs]

In [21]:
# with open(path_analysis_data_new + "psds_all_eloreta_45_adhoccov.pkl", "wb") as f:
#     pkl.dump(psds_all, f)

In [22]:
# with open(path_analysis_data_new + "psds_all_eloreta_45_adhoccov.pkl", "rb") as f:
#     psds_all = pkl.load(f)

In [23]:
psds_all.keys()

dict_keys(['sub01', 'sub03', 'sub04', 'sub07', 'sub10', 'sub12', 'sub09'])

In [24]:
# eeg_fooof_all = get_fooof(psds_all, n_channels=90, save_path=path_analysis_data_new + "fooof_all_source/")

### FOOOF on Source reconstructed PSDs

In [25]:
path_analysis_data_new

'D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/'

In [26]:
# import os
# from helpers.helper_functions import get_fooof

# fooof_save = path_analysis_data_new + "fooof_all_sources_vector_nofilt/"
# os.makedirs(fooof_save, exist_ok=True)
# eeg_fooof_all = get_fooof(psds_all, freq_range=(0.5, 45), save_path=fooof_save)

LOAD FOOOF

In [27]:
fooof_path = path_analysis_data_new + "fooof_all_sources_vector_nofilt/"
fooof_files = glob.glob(fooof_path + "*.pkl")


In [28]:
print(Path)

<class 'matplotlib.path.Path'>


In [29]:
from pathlib import Path as Path 

In [30]:


# eeg_fooof_all_subs = {sub:{} for sub in eeg_source_reconstructed_cont_cerebrum}
eeg_fooof_all_subs = {sub: {} for sub in psds_all}

for fl in fooof_files:
    file_stem = Path(fl).stem
    file_name = Path(fl).name

    sub_name = file_stem.split("_")[2][:5]
    parad_name = file_stem.split("_")[2][5:]

    try:
        parad_name_2 = file_stem.split("_")[3]
        parad_name += "_" + parad_name_2

    except:
        parad_name = parad_name
    
    print(sub_name, parad_name)
    
    with open(fl, "rb") as f:
        eeg_fooof_all_subs[sub_name][parad_name] = pkl.load(f)
        f.close()
        

sub01 rest_init
sub01 reward
sub03 rest_init
sub03 reward
sub04 rest_init
sub04 reward
sub07 music_segs
sub07 pink_segs
sub07 rest_end
sub07 rest_init
sub07 rest_mid
sub07 reward
sub09 music_segs
sub09 pink_noise
sub09 pink_segs
sub09 rest_end
sub09 rest_init
sub09 rest_mid
sub09 reward
sub10 music_segs
sub10 pink_noise
sub10 pink_segs
sub10 rest_init
sub10 rest_mid
sub10 reward
sub12 music_segs
sub12 pink_noise
sub12 pink_segs
sub12 rest_end
sub12 rest_init
sub12 rest_mid
sub12 reward


### Plot the Differences in FOOOF between Conditions

Import AAL Atlas

In [31]:
data_dir = path_analysis_data_new + "eeg_source/"

with open(f'{data_dir}AAL_labels.json', 'r') as f:
    aal_labels = json.load(f)

In [32]:
data_dir = path_analysis_data_new + "eeg_source/"
aal_nii_path = f'{data_dir}AAL.nii'   # adjust filename if needed
aal_img = nib.load(aal_nii_path)

aal_data = np.squeeze(np.asanyarray(aal_img.dataobj)).astype(int)
print("AAL Atlas Shape: ", aal_data.shape)

AAL Atlas Shape:  (91, 109, 91)


In [33]:
aal_regions = np.unique(aal_data)
aal_regions = aal_regions[aal_regions != 0]
aal_regions

data_dict = {
    "id": [],
    "regions": [],
    "labels": []
}

with open(path_analysis_data_new + "eeg_source/aal1_for_SPM8/ROI_MNI_V4.txt", "r") as f:
    for line in f:
        if line.strip():
            parts = line.split()  # split by whitespace
            
            data_dict["id"].append(parts[0])
            data_dict["regions"].append(parts[1])
            data_dict["labels"].append(parts[2])

print(data_dict.keys())


aal_regions_str = [str(reg) for reg in aal_regions]

count_reg_similar = 0

for reg in aal_regions_str:
    if reg in data_dict["labels"]:
        count_reg_similar += 1

print("Similar Regions Number: ", count_reg_similar)


label_region_mapping = {}

for indx in range(len(data_dict["labels"])):
    label_region_mapping[int(data_dict["labels"][indx])] = data_dict["regions"][indx]


dict_keys(['id', 'regions', 'labels'])
Similar Regions Number:  116


#### 1. Rest vs. Reward

In [34]:
eeg_fooof_rest_reward = {}


eeg_fooof_rest_reward["sub01"] = {}
eeg_fooof_rest_reward["sub01"]["rest_init"] = eeg_fooof_all_subs["sub01"]["rest_init"].copy()
eeg_fooof_rest_reward["sub01"]["reward"] = eeg_fooof_all_subs["sub01"]["reward"].copy()

eeg_fooof_rest_reward["sub03"] = {}
eeg_fooof_rest_reward["sub03"]["rest_init"] = eeg_fooof_all_subs["sub03"]["rest_init"].copy()
eeg_fooof_rest_reward["sub03"]["reward"] = eeg_fooof_all_subs["sub03"]["reward"].copy()

eeg_fooof_rest_reward["sub04"] = {}
eeg_fooof_rest_reward["sub04"]["rest_init"] = eeg_fooof_all_subs["sub04"]["rest_init"].copy()
eeg_fooof_rest_reward["sub04"]["reward"] = eeg_fooof_all_subs["sub04"]["reward"].copy()

eeg_fooof_rest_reward["sub07"] = {}
eeg_fooof_rest_reward["sub07"]["rest_init"] = eeg_fooof_all_subs["sub07"]["rest_init"].copy()
eeg_fooof_rest_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_reward["sub09"] = {} 
eeg_fooof_rest_reward["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_reward["sub10"] = {} 
eeg_fooof_rest_reward["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_rest_reward["sub12"] = {} 
eeg_fooof_rest_reward["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [35]:
eeg_fooof_rest_reward_matched = get_matched_legths(eeg_fooof_rest_reward)

Subject: sub01, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 80 epochs
(90, 223, 80) (90, 223, 80)
Subject: sub03, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 77 epochs
(90, 223, 77) (90, 223, 77)
Subject: sub04, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 35 epochs
(90, 223, 35) (90, 223, 35)
Subject: sub07, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 73 epochs
(90, 223, 73) (90, 223, 73)
Subject: sub09, Paradigms: ['rest_init', 'reward']
Truncated array1 from 88 → 67 epochs (dropped last 21)
(90, 223, 67) (90, 223, 67)
Subject: sub10, Paradigms: ['rest_init', 'reward']
Truncated array1 from 48 → 43 epochs (dropped last 5)
(90, 223, 43) (90, 223, 43)
Subject: sub12, Paradigms: ['rest_init', 'reward']
Truncated array1 from 70 → 53 epochs (dropped last 17)
(90, 223, 53) (90, 223, 53)


In [36]:
sub_id = "sub10"

# freqs = psds_all[sub_id]["rest_init"][1]
# psds = psds_all[sub_id]["rest_init"][0]
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/parads
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]   # only used for its region count (90) in get_fooof_bands

n_regions = 90

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}

_, rest_init_music_fooof_bands_pvalues_abs, fooof_rest_reward = get_channel_wise_fooof(freqs, eeg_fooof_rest_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals = get_fdr_p_values(rest_init_music_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)


sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
Paradigm 0 rest_init
Paradigm 1 reward
sub09
Paradigm 0 rest_init
Paradigm 1 reward
sub10
Paradigm 0 rest_init
Paradigm 1 reward
sub12
Paradigm 0 rest_init
Paradigm 1 reward


Split Half Comparison

In [29]:
def split_halves(entry):
    psd, ap, per, fr = entry
    h = per.shape[2] // 2
    return ([psd[..., :h],      ap[:, :h, :],      per[..., :h],      fr],
            [psd[..., h:2 * h], ap[:, h:2 * h, :], per[..., h:2 * h], fr])

def make_split(fooof_all, cond):
    out = {}
    for sub in fooof_all:
        if cond in fooof_all[sub]:
            h1, h2 = split_halves(fooof_all[sub][cond])
            out[sub] = {"half1": h1, "half2": h2}
    return out

def truncate_pair(pair, n_per_sub):
    return {sub: {c: [psd[..., :n_per_sub[sub]], ap[:, :n_per_sub[sub], :], per[..., :n_per_sub[sub]], fr]
                  for c, (psd, ap, per, fr) in d.items()}
            for sub, d in pair.items()}

def run_region_tests(pair, n=90):
    _, p_abs, _ = get_channel_wise_fooof(freqs, pair, bands, channels_n=n)
    fdr_per = get_fdr_p_values(p_abs, p_vals_per_band=n)
    fdr_ap = get_band_power_difference_all_subjects_significance_aperiodic_fdr(
        get_fooof_aperiodic(pair), pat_state, n_channels=n)
    return fdr_per, fdr_ap

def count_sig(fdr):
    return {sub: {k: int((np.asarray(v) < 0.05).sum()) for k, v in fdr[sub].items()} for sub in fdr}

split_music = make_split(eeg_fooof_all_subs, "reward")
split_rest  = make_split(eeg_fooof_all_subs, "rest_init")
n_rest_half = {sub: split_rest[sub]["half1"][2].shape[2] for sub in split_rest}
rm_trunc    = truncate_pair(eeg_fooof_rest_reward_matched, n_rest_half)

results_src = {
    "music half1 vs half2":          run_region_tests(split_music),
    "rest vs music (full N)":        run_region_tests(eeg_fooof_rest_reward_matched),
    "rest half1 vs half2":           run_region_tests(split_rest),
    "rest vs music (N = rest half)": run_region_tests(rm_trunc),
}
for name, (fdr_per, fdr_ap) in results_src.items():
    print(f"\n=== {name} === (significant regions out of 90)")
    per, ap = count_sig(fdr_per), count_sig(fdr_ap)
    for sub in per:
        print(f"  {sub}  periodic: {per[sub]}  | aperiodic: {ap.get(sub)}")
    nan = {s: int(sum(np.isnan(np.asarray(v, float)).sum() for v in list(fdr_per[s].values()) + list(fdr_ap[s].values()))) for s in fdr_per}
    print("  NaN p-values:", nan)

sub01
Paradigm 0 half1
Paradigm 1 half2
sub03
Paradigm 0 half1
Paradigm 1 half2
sub04
Paradigm 0 half1
Paradigm 1 half2
sub07
Paradigm 0 half1
Paradigm 1 half2
sub10
Paradigm 0 half1
Paradigm 1 half2
sub12
Paradigm 0 half1
Paradigm 1 half2
sub09
Paradigm 0 half1
Paradigm 1 half2
Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
Paradigm 0:  half1
Paradigm 1:  half2
sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
P

### Results summary (text) + split-half table

In [30]:
# ============================================================
# FULL REGION-LEVEL RESULTS PRINTOUT (self-contained, stats only)
# For every comparison: per subject and band/parameter, the number of
# significant regions (of 90, FDR per subject) and, among those, how many
# are higher (+) / lower (-) in the second-named condition.
# Writes the output to a text file as well.
# ============================================================
import io as _io

def _epoch_mean(a):
    a = np.asarray(a, float)
    return np.nanmean(a, axis=1) if a.ndim == 2 else a

def _pair_r(subs, c1, c2):
    return get_matched_legths({s: {c1: eeg_fooof_all_subs[s][c1].copy(),
                                   c2: eeg_fooof_all_subs[s][c2].copy()} for s in subs})

def _region_report(buf, name, pair, c1, c2):
    fdr_per, fdr_ap = run_region_tests(pair)
    _, pow_abs = get_fooof_bands(pair, freqs, psds, bands)
    ap_vals = get_fooof_aperiodic(pair)
    n_ep = {s: pair[s][c1][2].shape[2] for s in pair}
    print(f"\n=== {name}  [{c1} -> {c2}] ===", file=buf)
    print("epochs per condition:", {s: n_ep[s] for s in sorted(n_ep)}, file=buf)
    for s in sorted(fdr_per):
        for fdr, vals, keys in ((fdr_per, pow_abs, list(bands)),
                                (fdr_ap, ap_vals, ["Offset", "Exponent"])):
            cells = []
            for k in keys:
                if k not in fdr[s]:
                    continue
                p = np.asarray(fdr[s][k], float)
                sig = p < 0.05
                d = _epoch_mean(vals[s][c2][k]) - _epoch_mean(vals[s][c1][k])
                
                pos, neg = int((sig & (d > 0)).sum()), int((sig & (d < 0)).sum())
                cells.append(f"{k[:5]}:{int(sig.sum()):2d}(+{pos}/-{neg})")
            print(f"  {s}  " + "  ".join(cells), file=buf)

buf = _io.StringIO()
print("REGION-LEVEL RESULTS SUMMARY (significant regions of 90, FDR per subject;",
      "sign = epoch-mean(condition2) - epoch-mean(condition1))", file=buf)

subs_all, subs_seg, subs_cont = (["sub01", "sub03", "sub04", "sub07", "sub09", "sub10", "sub12"],
                                 ["sub07", "sub09", "sub10", "sub12"],
                                 ["sub09", "sub10", "sub12"])

rr = _pair_r(subs_all, "rest_init", "reward")
_region_report(buf, "1. initial rest vs continuous music (full N)", rr, "rest_init", "reward")
_region_report(buf, "2. segmented pink noise vs segmented music",
               _pair_r(subs_seg, "pink_segs", "music_segs"), "pink_segs", "music_segs")
_region_report(buf, "3. continuous pink noise vs continuous music",
               _pair_r(subs_cont, "pink_noise", "reward"), "pink_noise", "reward")
_region_report(buf, "4. intermediate rest vs segmented music",
               _pair_r(subs_seg, "rest_mid", "music_segs"), "rest_mid", "music_segs")
_region_report(buf, "5. intermediate rest vs segmented pink noise",
               _pair_r(subs_seg, "rest_mid", "pink_segs"), "rest_mid", "pink_segs")
_region_report(buf, "6. intermediate rest vs continuous music",
               _pair_r(subs_seg, "rest_mid", "reward"), "rest_mid", "reward")

_region_report(buf, "7. SPLIT music half1 vs half2", make_split(eeg_fooof_all_subs, "reward"), "half1", "half2")
split_rest_r = make_split(eeg_fooof_all_subs, "rest_init")
_region_report(buf, "8. SPLIT rest half1 vs half2", split_rest_r, "half1", "half2")
n_rh = {s: split_rest_r[s]["half1"][2].shape[2] for s in split_rest_r}
_region_report(buf, "9. rest vs music (N = rest half)", truncate_pair(rr, n_rh), "rest_init", "reward")

text = buf.getvalue()
print(text)
out_file = plots_path + "/region_level_results_summary.txt"
with open(out_file, "w", encoding="utf-8") as f:
    f.write(text)
print("\nsaved ->", out_file)

Subject: sub01, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 80 epochs
(90, 223, 80) (90, 223, 80)
Subject: sub03, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 77 epochs
(90, 223, 77) (90, 223, 77)
Subject: sub04, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 35 epochs
(90, 223, 35) (90, 223, 35)
Subject: sub07, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 73 epochs
(90, 223, 73) (90, 223, 73)
Subject: sub09, Paradigms: ['rest_init', 'reward']
Truncated array1 from 88 → 67 epochs (dropped last 21)
(90, 223, 67) (90, 223, 67)
Subject: sub10, Paradigms: ['rest_init', 'reward']
Truncated array1 from 48 → 43 epochs (dropped last 5)
(90, 223, 43) (90, 223, 43)
Subject: sub12, Paradigms: ['rest_init', 'reward']
Truncated array1 from 70 → 53 epochs (dropped last 17)
(90, 223, 53) (90, 223, 53)
sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
Pa

In [31]:
print(split_half_latex_table(results_src, "region", list(bands), 90,
                             "tab:split_half_region", pat_state=pat_state_dict))

\begingroup
\small
\setlength{\tabcolsep}{4pt}
\begin{xltabular}{\textwidth}{@{}l X r r r r r r r@{}}
\caption{Split-half control at the region level. Number of significant regions (of 90) per subject, frequency band, and aperiodic parameter, for the initial rest versus continuous music comparison and for the within-recording split-half comparisons (H1, H2: first and second half of the recording). The rest versus music comparison is shown with all epochs and with its epoch count reduced to that of the rest halves. Bold: rest-versus-music count exceeds the corresponding split-half count in both pairings.}
\label{tab:split_half_region}\\
\toprule
\textbf{Subject} & \textbf{Comparison} & $\delta$ & $\theta$ & $\alpha$ & $\beta$ & $\gamma$ & Offset & Exp. \\
\midrule
\endhead
\multirow{4}{*}{SUB1 (UWS)} & Initial rest vs music & \textbf{13} & \textbf{15} & \textbf{43} & \textbf{70} & \textbf{52} & \textbf{41} & \textbf{58} \\
 & Music H1 vs H2 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & Rest vs musi

In [25]:
mcs_minus_subjects = ["sub04", "sub07", "sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_rest_reward,
    mcs_minus_subjects,
    condition1="rest_init",
    condition2="reward",   # confirm this matches your actual music-condition key
    bands=bands,
    abs_rel=0,
    n_boot=1000,
    n_channels=90
)


for band in bands:
    print(f"{band}: {sig_mask[band].sum()} / 90 channels significant at group level")


# First comparison — produces the full table opening
latex_code_1 = latex_group_vs_individual_table(
    fdr_corrected_p_vals, mcs_minus_subjects, sig_mask, bands,
    comparison_label="Rest vs. Continuous Music",
    mode='start',
    n_channels=90,
    caption="Group-level versus individual-level significance across source regions and frequency bands.",
    label="tab:group_vs_individual_source",
)


print(latex_code_1)

delta: 0 / 90 channels significant at group level
theta: 0 / 90 channels significant at group level
alpha: 0 / 90 channels significant at group level
beta: 0 / 90 channels significant at group level
gamma low: 0 / 90 channels significant at group level
\begin{table}[htbp]
\centering
\caption{Group-level versus individual-level significance across source regions and frequency bands.}
\label{tab:group_vs_individual_source}

\begin{tabularx}{\textwidth}{@{}l l X X X X@{}}
\toprule
\textbf{Comparison} &
\textbf{Band} &
\textbf{Group-level sig.} &
\textbf{Sig. in $\geq$1 subject} &
\textbf{Sig. in majority} &
\textbf{Sig. in all subjects} \\
\midrule

\multirow{5}{*}{\shortstack[l]{Rest vs.\\Continuous Music}}
& $\delta$ & 0/90 & 87/90 & 2/90 ($\geq$3/5) & 0/90 \\
& $\theta$ & 0/90 & 90/90 & 15/90 ($\geq$3/5) & 0/90 \\
& $\alpha$ & 0/90 & 89/90 & 48/90 ($\geq$3/5) & 0/90 \\
& $\beta$ & 0/90 & 89/90 & 30/90 ($\geq$3/5) & 0/90 \\
& $\gamma$ & 0/90 & 90/90 & 62/90 ($\geq$3/5) & 0/90 \\

\botto

In [37]:
_, eeg_fooof_bands_abs_rest_init_music = get_fooof_bands(eeg_fooof_rest_reward_matched, freqs, psds, bands)


In [38]:
for sub in eeg_fooof_bands_abs_rest_init_music:
    for cond in eeg_fooof_bands_abs_rest_init_music[sub]:
        print(sub, cond, {b: f"{np.nanmin(v):.3f}" for b, v in eeg_fooof_bands_abs_rest_init_music[sub][cond].items()})

sub01 rest_init {'delta': '0.001', 'theta': '0.043', 'alpha': '-0.051', 'beta': '3.350', 'gamma low': '0.516'}
sub01 reward {'delta': '-0.075', 'theta': '0.006', 'alpha': '-0.047', 'beta': '3.625', 'gamma low': '0.510'}
sub03 rest_init {'delta': '0.507', 'theta': '0.292', 'alpha': '0.203', 'beta': '0.990', 'gamma low': '3.273'}
sub03 reward {'delta': '0.325', 'theta': '0.805', 'alpha': '0.425', 'beta': '0.939', 'gamma low': '2.511'}
sub04 rest_init {'delta': '0.515', 'theta': '-0.025', 'alpha': '-0.123', 'beta': '1.828', 'gamma low': '1.099'}
sub04 reward {'delta': '0.504', 'theta': '0.240', 'alpha': '0.050', 'beta': '1.474', 'gamma low': '1.138'}
sub07 rest_init {'delta': '0.460', 'theta': '0.533', 'alpha': '0.960', 'beta': '1.240', 'gamma low': '0.446'}
sub07 reward {'delta': '0.531', 'theta': '0.887', 'alpha': '1.298', 'beta': '1.424', 'gamma low': '0.146'}
sub09 rest_init {'delta': '0.042', 'theta': '0.311', 'alpha': '-0.118', 'beta': '1.166', 'gamma low': '0.329'}
sub09 reward {'d

In [39]:
subjects_sorted = sorted(fdr_corrected_p_vals.keys())
sub_label_dict = {sub: f"SUB{i+1}" for i, sub in enumerate(subjects_sorted)}
sub_label_dict

{'sub01': 'SUB1',
 'sub03': 'SUB2',
 'sub04': 'SUB3',
 'sub07': 'SUB4',
 'sub09': 'SUB5',
 'sub10': 'SUB6',
 'sub12': 'SUB7'}

In [40]:
eeg_fooof_bands_abs_rest_init_music["sub01"].keys()

dict_keys(['rest_init', 'reward'])

In [37]:
def plot_fooof_fdr_results(fdr_corrected_p_vals, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color,view, pat_state_dict, save_dir1, 
                           save_dir2, sub_label_dict=None, out_filename=None, transpose=False, show_sub_labels=True):
    
    for sub_id in fdr_corrected_p_vals.keys():
        for band in bands:
            plot_significance_single_sub_band(
                fdr_corrected_p_vals=fdr_corrected_p_vals,
                aal_img=aal_img,
                label_region_mapping=label_region_mapping,
                sub_id=sub_id,
                band=band,
                parad1=parad1,
                parad2=parad2,
                view=view,
                pvals=pvals,
                cmap_color=cmap_color,
                save_dir=save_dir1,
            )

    build_vector_grid_pdf(
        image_dir=save_dir1,
        parad1=parad1,
        parad2=parad2,
        bands=bands,  # controls row order
        pat_state=pat_state_dict,
        save_dir=save_dir2,
        sub_labels=sub_label_dict,
        
        out_filename=out_filename,
        transpose=transpose,
        show_sub_labels=show_sub_labels
    )




In [32]:
save_path1 = f"{plots_path}/fdr_source/rest_reward/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/rest_reward/fdr_plots/merged/"

parad1 = "Rest"
parad2 = "Music Cont."
view = "lateral"

# cmap_color = ListedColormap(['#DAE8EC', "#D45E00"])  # warm greige / muted amber
cmap_color = ListedColormap(["#EDF4F6", "#7DB17B"])   # non-sig pale / significant green, pencil-matchedpvals = True 

legend_label1 = "Rest"
legend_label2 = "Music Cont."
pvals=True
plot_fooof_fdr_results(fdr_corrected_p_vals, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, pat_state_dict, 
                       save_path1, save_path2, out_filename="src_sig_rest_musiccont.pdf", transpose=False, show_sub_labels=True)  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/fdr_plots/sub03_Rest

In [38]:
def plot_fooof_relative_difference_results(power_dict, bands, aal_img, label_region_mapping, condition1, condition2, cmap_color, view, pat_state_list, 
                                           save_dir1, save_dir2, sub_label_dict=None, out_filename=None, transpose=False, show_sub_labels=True, difference="absolute"):

    for sub_id in power_dict.keys():
        for band in bands:
            plot_relative_difference_single_sub_band(
                power_dict=power_dict,
                aal_img=aal_img,
                label_region_mapping=label_region_mapping,
                cmap_color=cmap_color,
                sub_id=sub_id,
                band=band,
                condition1=condition1,
                condition2=condition2,
                view=view,
                save_dir=save_dir1, difference=difference)

    build_vector_grid_pdf(
        image_dir=save_dir1,
        parad1=condition1,
        parad2=condition2,
        bands=bands,
        pat_state=pat_state_list,
        save_dir=save_dir2,
        sub_labels=sub_label_dict,
        out_filename=out_filename, transpose=transpose, show_sub_labels=show_sub_labels)





In [34]:

_, eeg_fooof_bands_abs_rest_init_music = get_fooof_bands(eeg_fooof_rest_reward_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_rest_init_music,
    bands, aal_img, label_region_mapping,
    condition1="rest_init", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/rest_reward/color/",
    save_dir2=f"{plots_path}/fdr_source/rest_reward/color/merged/",
    out_filename="src_reldiff_rest_musiccont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/color/sub01_rest_init_reward_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/color/sub01_rest_init_reward_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/color/sub01_rest_init_reward_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/color/sub01_rest_init_reward_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/color/sub01_rest_init_reward_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/color/sub03_rest_init_reward_delta.pdf
S

In [35]:
subjects_sorted = sorted(fdr_corrected_p_vals.keys())
sub_label_dict = {sub: f"SUB{i+1}" for i, sub in enumerate(subjects_sorted)}
sub_label_dict

_, eeg_fooof_bands_abs_rest_init_music = get_fooof_bands(eeg_fooof_rest_reward_matched, freqs, psds, bands)

summary = build_cluster_summary_all(
    fdr_corrected_p_vals, eeg_fooof_bands_abs_rest_init_music,
    condition1="rest_init", condition2="reward", bands=bands,
    label_region_mapping=label_region_mapping,
    region_to_anatomical_lobe=region_to_anatomical_lobe, difference="absolute"
)

# plot_cluster_bubble_heatmap(
#     summary, bands,
#     cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
#     pat_state=pat_state_dict, sub_labels=sub_label_dict,
#     condition1_label="Rest", condition2_label="Music Cont.",
#     save_path=f"{plots_path}/fdr_source/rest_reward/color/bubble/src_bubble_rest_musiccont.pdf",
# )

# plot_cluster_bubble_heatmap(summary, bands=bands, cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
#                             pat_state=pat_state_dict, sub_labels=sub_label_dict,
#                             vertical=True, show_sub_labels=True,
#                             save_path=f"{plots_path}/fdr_source/rest_reward/color/bubble/src_bubble_rest_musiccont.pdf")


plot_cluster_bubble_heatmap(
    summary,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    cluster_order=["Frontal", "Parietal", "Temporal", "Occipital",
                   "Insula", "Cingulate", "Limbic", "SCGM"],
    pat_state=pat_state_dict,
    sub_labels=sub_label_dict,
    vertical=False,
    show_sub_labels=True,
    abbrev_clusters=True,
    cbar_location="bottom",
    save_path=plots_path + "/fdr_source/rest_reward/color/bubble/src_bubble_rest_musiccont.pdf",
)


Aperiodic

In [39]:
eeg_fooof_rest_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_reward_matched)
fdr_corrected_p_vals_rest_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_reward_aperiodic, 
                                                                                                               pat_state, n_channels=n_regions)


Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward


In [43]:
subjects_sorted = sorted(fdr_corrected_p_vals_rest_reward_aperiodic.keys())
sub_label_dict = {sub: f"SUB{i+1}" for i, sub in enumerate(subjects_sorted)}
sub_label_dict

{'sub01': 'SUB1',
 'sub03': 'SUB2',
 'sub04': 'SUB3',
 'sub07': 'SUB4',
 'sub09': 'SUB5',
 'sub10': 'SUB6',
 'sub12': 'SUB7'}

In [38]:
# %autoreload now

summary = build_cluster_summary_all(
    fdr_corrected_p_vals_rest_reward_aperiodic, eeg_fooof_rest_reward_aperiodic,
    condition1="rest_init", condition2="reward", bands=["Offset", "Exponent"],
    label_region_mapping=label_region_mapping,
    region_to_anatomical_lobe=region_to_anatomical_lobe, difference="absolute",
)


plot_cluster_bubble_heatmap(
    summary,
    bands=["Offset", "Exponent"],
    cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
    pat_state=pat_state_dict,
    sub_labels=sub_label_dict,
    vertical=False,
    show_sub_labels=True,
    abbrev_clusters=True,
    per_band_cnorm=True,
    cbar_location="bottom",
    cbar_label=r"Mean $\Delta$, significant regions",
    save_path=plots_path + "/fdr_source/rest_reward/aperiodic_color/bubble/src_aperiodic_bubble_rest_musiccont.pdf",
)


In [39]:
aperiodic_params = ["Exponent", "Offset"] 
save_path1 = f"{plots_path}/fdr_source/rest_reward/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/rest_reward/aperiodic/merged/"

parad1 = "rest"
parad2 = "reward"
view = "lateral"
# cmap_color = ListedColormap(['#DAE8EC', "#D45E00"])  # warm greige / muted amber
pvals = True 

plot_fooof_fdr_results(fdr_corrected_p_vals_rest_reward_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, 
                       view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_rest_musiccont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic/sub01_rest_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic/sub01_rest_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic/sub03_rest_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic/sub03_rest_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic/sub04_rest_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic/sub04_rest_reward_Offset.pd

In [40]:
bands = ["Exponent", "Offset"]  # Aperiodic parameters  
plot_fooof_relative_difference_results(
    eeg_fooof_rest_reward_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="rest_init", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/rest_reward/aperiodic_color/",
    save_dir2=f"{plots_path}/fdr_source/rest_reward/aperiodic_color/merged/",
    out_filename="src_aperiodic_reldiff_rest_musiccont.pdf", difference="absolute")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic_color/sub01_rest_init_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic_color/sub01_rest_init_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic_color/sub03_rest_init_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic_color/sub03_rest_init_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/aperiodic_color/sub04_rest_init_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_s

#### 2. Music Segs vs. Pink Segs

In [41]:
eeg_fooof_all_subs["sub07"].keys()

dict_keys(['music_segs', 'pink_segs', 'rest_end', 'rest_init', 'rest_mid', 'reward'])

In [42]:
eeg_fooof_music_pink_segs = {}

eeg_fooof_music_pink_segs["sub07"] = {}
eeg_fooof_music_pink_segs["sub07"]["pink_segs"] = eeg_fooof_all_subs["sub07"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub07"]["music_segs"] = eeg_fooof_all_subs["sub07"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub09"] = {} 
eeg_fooof_music_pink_segs["sub09"]["pink_segs"] = eeg_fooof_all_subs["sub09"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub09"]["music_segs"] = eeg_fooof_all_subs["sub09"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub10"] = {} 
eeg_fooof_music_pink_segs["sub10"]["pink_segs"] = eeg_fooof_all_subs["sub10"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub10"]["music_segs"] = eeg_fooof_all_subs["sub10"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub12"] = {} 
eeg_fooof_music_pink_segs["sub12"]["pink_segs"] = eeg_fooof_all_subs["sub12"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub12"]["music_segs"] = eeg_fooof_all_subs["sub12"]["music_segs"].copy()

eeg_fooof_music_pink_segs_matched = get_matched_legths(eeg_fooof_music_pink_segs)

Subject: sub07, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 128 → 125 epochs (dropped last 3)
(90, 223, 125) (90, 223, 125)
Subject: sub09, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 94 → 93 epochs (dropped last 1)
(90, 223, 93) (90, 223, 93)
Subject: sub10, Paradigms: ['pink_segs', 'music_segs']
Truncated array2 from 68 → 35 epochs (dropped last 33)
(90, 223, 35) (90, 223, 35)
Subject: sub12, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 83 → 82 epochs (dropped last 1)
(90, 223, 82) (90, 223, 82)


In [43]:
bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}

In [44]:
sub_id = "sub10"

# freqs = psds_all[sub_id]["rest_init"][1]
# psds = psds_all[sub_id]["rest_init"][0]
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/parads
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]   # only used for its region count (90) in get_fooof_bands

n_regions = 90

_, music_pink_seg_fooof_bands_pvalues_abs, fooof_music_pink_segs = get_channel_wise_fooof(freqs, eeg_fooof_music_pink_segs_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_music_pink_segs = get_fdr_p_values(music_pink_seg_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)


sub07
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub09
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub10
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub12
Paradigm 0 pink_segs
Paradigm 1 music_segs


In [45]:
mcs_minus_subjects = ["sub07", "sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_music_pink_segs,
    mcs_minus_subjects,
    condition1="pink_segs",
    condition2="music_segs",   # confirm this matches your actual music-condition key
    bands=bands,
    abs_rel=0,
    n_boot=1000,
    n_channels=90

)

for band in bands:
    print(f"{band}: {sig_mask[band].sum()} 90 channels significant at group level")


latex_code_2 = latex_group_vs_individual_table(
    fdr_corrected_p_vals_music_pink_segs, mcs_minus_subjects, sig_mask, bands,
    comparison_label="Segmented Music vs. Segmented Pink Noise",
    mode='append',
    n_channels=90
)

print(latex_code_2)




delta: 0 90 channels significant at group level
theta: 0 90 channels significant at group level
alpha: 0 90 channels significant at group level
beta: 0 90 channels significant at group level
gamma low: 0 90 channels significant at group level
\midrule

\multirow{5}{*}{\shortstack[l]{Segmented Music vs.\\Segmented Pink Noise}}
& $\delta$ & 0/90 & 44/90 & 0/90 ($\geq$2/4) & 0/90 \\
& $\theta$ & 0/90 & 19/90 & 0/90 ($\geq$2/4) & 0/90 \\
& $\alpha$ & 0/90 & 13/90 & 0/90 ($\geq$2/4) & 0/90 \\
& $\beta$ & 0/90 & 3/90 & 0/90 ($\geq$2/4) & 0/90 \\
& $\gamma$ & 0/90 & 1/90 & 0/90 ($\geq$2/4) & 0/90 \\




In [46]:
save_path1 = f"{plots_path}/fdr_source/music_pink_segs/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/music_pink_segs/fdr_plots/merged/"

parad1 = "Pink Noise Segs"
parad2 = "Music Segs"
view = "lateral"

# cmap_color = ListedColormap(['#DAE8EC', "#D45E00"])  # warm greige / muted amber
pvals = True 
legend_label1 = "Pink Noise Segs"
legend_label2 = "Music Segs"

plot_fooof_fdr_results(fdr_corrected_p_vals_music_pink_segs, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, 
                       pat_state_dict, save_path1, save_path2, out_filename="src_sig_musicsegs_pinksegs.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/ana

In [47]:
save_path1 = f"{plots_path}/fdr_source/music_pink_segs/color/"
save_path2 = f"{plots_path}/fdr_source/music_pink_segs/color/merged/"

# parad1 = "Music Segs"
# parad2 = "Pink Noise Segs"
view = "lateral"
# cmap_color = ListedColormap(['white', 'black'])
pvals = True 

_, eeg_fooof_bands_abs_pink_music = get_fooof_bands(eeg_fooof_music_pink_segs_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_pink_music,
    bands, aal_img, label_region_mapping,
    condition1="pink_segs", condition2="music_segs",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2, out_filename="src_reldiff_musicsegs_pinksegs.pdf")




Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_s

In [48]:
_, eeg_fooof_bands_abs_pink_music = get_fooof_bands(eeg_fooof_music_pink_segs_matched, freqs, psds, bands)

summary = build_cluster_summary_all(
    fdr_corrected_p_vals_music_pink_segs, eeg_fooof_bands_abs_pink_music,
    condition1="pink_segs", condition2="music_segs", bands=bands,
    label_region_mapping=label_region_mapping,
    region_to_anatomical_lobe=region_to_anatomical_lobe, difference="absolute"
)

# plot_cluster_bubble_heatmap(
#     summary, bands,
#     cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
#     pat_state=pat_state_dict, sub_labels=sub_label_dict,
#     condition1_label="Pink Noise Seg.", condition2_label="Music Seg.",
#     save_path=f"{plots_path}/fdr_source/music_pink_segs/color/bubble/src_bubble_musicsegs_pinksegs.pdf",
# )


plot_cluster_bubble_heatmap(
    summary,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    cluster_order=["Frontal", "Parietal", "Temporal", "Occipital",
                   "Insula", "Cingulate", "Limbic", "SCGM"],
    pat_state=pat_state_dict,
    sub_labels=sub_label_dict,
    vertical=False,
    show_sub_labels=True,
    abbrev_clusters=True,
    cbar_location="bottom",
    save_path=f"{plots_path}/fdr_source/music_pink_segs/color/bubble/src_bubble_musicsegs_pinksegs.pdf")


Aperiodic

In [49]:
eeg_fooof_music_pink_segs_aperiodic = get_fooof_aperiodic(eeg_fooof_music_pink_segs_matched)
fdr_corrected_p_vals_music_pink_segs_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_music_pink_segs_aperiodic, pat_state, n_channels=n_regions)

Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs


In [50]:
aperiodic_params = ["Exponent", "Offset"] 
save_path1 = f"{plots_path}/fdr_source/music_pink_segs/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/music_pink_segs/aperiodic/merged/"

parad1 = "Pink Noise Segs"
parad2 = "Music Segs"
view = "lateral"
# cmap_color = ListedColormap(['#DAE8EC', "#D45E00"])  # warm greige / muted amber
pvals = True 

plot_fooof_fdr_results(fdr_corrected_p_vals_music_pink_segs_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, 
                       cmap_color, view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_musicsegs_pinksegs.pdf")  



Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic/sub07_Pink Noise Segs_Music Segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic/sub07_Pink Noise Segs_Music Segs_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic/sub09_Pink Noise Segs_Music Segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic/sub09_Pink Noise Segs_Music Segs_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic/sub10_Pink Noise Segs_Music Segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_s

In [51]:
bands = ["Exponent", "Offset"]  # Aperiodic parameters

save_path1 = f"{plots_path}/fdr_source/music_pink_segs/aperiodic_color/"
save_path2 = f"{plots_path}/fdr_source/music_pink_segs/aperiodic_color/merged/"


plot_fooof_relative_difference_results(
    eeg_fooof_music_pink_segs_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="pink_segs", condition2="music_segs",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2, out_filename="src_aperiodic_reldiff_musicsegs_pinksegs.pdf", difference="absolute")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic_color/sub07_pink_segs_music_segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic_color/sub07_pink_segs_music_segs_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic_color/sub09_pink_segs_music_segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic_color/sub09_pink_segs_music_segs_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/aperiodic_color/sub10_pink_segs_music_segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_s

#### 3. Rest vs. Pink Noise

In [52]:
eeg_fooof_all_subs["sub07"].keys()

dict_keys(['music_segs', 'pink_segs', 'rest_end', 'rest_init', 'rest_mid', 'reward'])

In [53]:
eeg_fooof_rest_pink = {}

eeg_fooof_rest_pink["sub09"] = {} 
eeg_fooof_rest_pink["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_pink["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()

eeg_fooof_rest_pink["sub10"] = {} 
eeg_fooof_rest_pink["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_pink["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()

eeg_fooof_rest_pink["sub12"] = {}
eeg_fooof_rest_pink["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_pink["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()

eeg_fooof_rest_pink_matched = get_matched_legths(eeg_fooof_rest_pink)

Subject: sub09, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 88 → 51 epochs (dropped last 37)
(90, 223, 51) (90, 223, 51)
Subject: sub10, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 48 → 36 epochs (dropped last 12)
(90, 223, 36) (90, 223, 36)
Subject: sub12, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 70 → 58 epochs (dropped last 12)
(90, 223, 58) (90, 223, 58)


In [54]:
sub_id = "sub10"

# freqs = psds_all[sub_id]["rest_init"][1]
# psds = psds_all[sub_id]["rest_init"][0]

freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/parads
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]   # only used for its region count (90) in get_fooof_bands

n_regions = 90

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}

_, rest_pink_fooof_bands_pvalues_abs, _ = get_channel_wise_fooof(freqs, eeg_fooof_rest_pink_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_rest_pink = get_fdr_p_values(rest_pink_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)


sub09
Paradigm 0 rest_init
Paradigm 1 pink_noise
sub10
Paradigm 0 rest_init
Paradigm 1 pink_noise
sub12
Paradigm 0 rest_init
Paradigm 1 pink_noise


In [55]:
save_path1 = f"{plots_path}/fdr_source/rest_pink_noise/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/rest_pink_noise/fdr_plots/merged/"

parad1 = "Rest"
parad2 = "Pink Noise"
view = "lateral"
pvals = True 


legend_label1 = "Rest"
legend_label2 = "Pink Noise Cont."
plot_fooof_fdr_results(fdr_corrected_p_vals_rest_pink, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, pat_state_dict, save_path1, 
                       save_path2, out_filename="src_sig_rest_pinkcont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/f

In [56]:
view = "lateral"
pvals = True 

_, eeg_fooof_bands_abs_rest_pink = get_fooof_bands(eeg_fooof_rest_pink_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_rest_pink,
    bands, aal_img, label_region_mapping,
    condition1="rest_init", condition2="pink_noise",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/rest_pink_noise/color/",
    save_dir2=f"{plots_path}/fdr_source/rest_pink_noise/color/merged/", out_filename="src_reldiff_rest_pinkcont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_no

In [57]:
# _, eeg_fooof_bands_abs_rest_pink = get_fooof_bands(eeg_fooof_rest_pink_matched, freqs, psds, bands)

# summary = build_cluster_summary_all(
#     fdr_corrected_p_vals_rest_pink, eeg_fooof_bands_abs_rest_pink,
#     condition1="pink_segs", condition2="music_segs", bands=bands,
#     label_region_mapping=label_region_mapping,
#     region_to_anatomical_lobe=region_to_anatomical_lobe,
# )

# plot_cluster_bubble_heatmap(
#     summary, bands,
#     cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
#     pat_state=pat_state_dict, sub_labels=sub_label_dict,
#     condition1_label="Pink Noise Cont.", condition2_label="Music Cont.",
#     save_path=f"{plots_path}/fdr_source/rest_pink_noise/color/bubble/cluster_bubble_rest_music.pdf",
# )


Aperiodic

In [58]:
eeg_fooof_rest_pink_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_pink_matched)


fdr_corrected_p_vals_rest_pink_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_pink_aperiodic, pat_state, n_channels=n_regions)



aperiodic_params = ["Exponent", "Offset"] 

save_path1 = f"{plots_path}/fdr_source/rest_pink_noise/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/rest_pink_noise/aperiodic/merged/"

parad1 = "Rest"
parad2 = "Pink Noise"
view = "lateral"
pvals = True 


plot_fooof_fdr_results(fdr_corrected_p_vals_rest_pink_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, 
                       view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_rest_pinkcont.pdf")  




Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic/sub09_Rest_Pink Noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic/sub09_Rest_Pink Noise_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic/sub10_Rest_Pink Noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic/sub10

In [59]:
bands = ["Exponent", "Offset"]  # Aperiodic parameters

save_path1 = f"{plots_path}/fdr_source/rest_pink_noise/aperiodic_color/"
save_path2 = f"{plots_path}/fdr_source/rest_pink_noise/aperiodic_color/merged/"


plot_fooof_relative_difference_results(
    eeg_fooof_rest_pink_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="rest_init", condition2="pink_noise",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2,out_filename="src_aperiodic_reldiff_rest_pinkcont.pdf", difference="absolute")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic_color/sub09_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic_color/sub09_rest_init_pink_noise_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic_color/sub10_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic_color/sub10_rest_init_pink_noise_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/aperiodic_color/sub12_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_s

#### 4. Reward vs. Pink Noise

In [60]:
eeg_fooof_pink_noise_reward = {}

eeg_fooof_pink_noise_reward["sub09"] = {} 
eeg_fooof_pink_noise_reward["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub10"] = {} 
eeg_fooof_pink_noise_reward["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub12"] = {}
eeg_fooof_pink_noise_reward["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

eeg_fooof_pink_noise_reward_matched = get_matched_legths(eeg_fooof_pink_noise_reward)

Subject: sub09, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 67 → 51 epochs (dropped last 16)
(90, 223, 51) (90, 223, 51)
Subject: sub10, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 43 → 36 epochs (dropped last 7)
(90, 223, 36) (90, 223, 36)
Subject: sub12, Paradigms: ['pink_noise', 'reward']
Truncated array1 from 58 → 53 epochs (dropped last 5)
(90, 223, 53) (90, 223, 53)


In [61]:
sub_id = "sub10"

# freqs = psds_all[sub_id]["rest_init"][1]
# psds = psds_all[sub_id]["rest_init"][0]

freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/parads
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]   # only used for its region count (90) in get_fooof_bands

n_regions = 90

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}


_, pink_noise_reward_fooof_bands_pvalues_abs, fooof_pink_noise_reward = get_channel_wise_fooof(freqs, eeg_fooof_pink_noise_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_pink_noise_reward = get_fdr_p_values(pink_noise_reward_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)

sub09
Paradigm 0 pink_noise
Paradigm 1 reward
sub10
Paradigm 0 pink_noise
Paradigm 1 reward
sub12
Paradigm 0 pink_noise
Paradigm 1 reward


In [62]:
mcs_minus_subjects = ["sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_pink_noise_reward,
    mcs_minus_subjects,
    condition1="pink_noise",
    condition2="reward",   # confirm this matches your actual music-condition key
    bands=bands,
    abs_rel=0,
    n_boot=1000,
    n_channels=90
)

for band in bands:
    print(f"{band}: {sig_mask[band].sum()} / 90 channels significant at group level")


latex_code_2 = latex_group_vs_individual_table(
    fdr_corrected_p_vals_pink_noise_reward, mcs_minus_subjects, sig_mask, bands,
    comparison_label="Continuous Music vs. Continuous Pink Noise",
    mode='append',
    n_channels=90
)

print(latex_code_2)




delta: 0 / 90 channels significant at group level
theta: 0 / 90 channels significant at group level
alpha: 0 / 90 channels significant at group level
beta: 0 / 90 channels significant at group level
gamma low: 0 / 90 channels significant at group level
\midrule

\multirow{5}{*}{\shortstack[l]{Continuous Music vs.\\Continuous Pink Noise}}
& $\delta$ & 0/90 & 0/90 & 0/90 ($\geq$2/3) & 0/90 \\
& $\theta$ & 0/90 & 1/90 & 0/90 ($\geq$2/3) & 0/90 \\
& $\alpha$ & 0/90 & 0/90 & 0/90 ($\geq$2/3) & 0/90 \\
& $\beta$ & 0/90 & 0/90 & 0/90 ($\geq$2/3) & 0/90 \\
& $\gamma$ & 0/90 & 7/90 & 0/90 ($\geq$2/3) & 0/90 \\




In [63]:
save_path1 = f"{plots_path}/fdr_source/pink_noise_reward/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/pink_noise_reward/fdr_plots/merged/"

parad1 = "Pink Noise"
parad2 = "Music Cont."
view = "lateral"
pvals = True

legend_label1 = "Pink Noise Cont."
legend_label2 = "Music Cont."

plot_fooof_fdr_results(fdr_corrected_p_vals_pink_noise_reward, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, 
                       pat_state_dict, save_path1, save_path2, out_filename="src_sig_pinkcont_musiccont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plot

In [64]:

parad1 = "Pink Noise Cont."
parad2 = "Music Cont."
view = "lateral"

_, eeg_fooof_bands_abs_rest_pink = get_fooof_bands(eeg_fooof_pink_noise_reward_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_rest_pink,
    bands, aal_img, label_region_mapping,
    condition1="pink_noise", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/pink_noise_reward/color/",
    save_dir2=f"{plots_path}/fdr_source/pink_noise_reward/color/merged/", out_filename="src_reldiff_pinkcont_musiccont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward

In [65]:
_, eeg_fooof_bands_abs_pink_reward = get_fooof_bands(eeg_fooof_pink_noise_reward_matched, freqs, psds, bands)


summary = build_cluster_summary_all(
    fdr_corrected_p_vals_pink_noise_reward, eeg_fooof_bands_abs_pink_reward,
    condition1="pink_noise", condition2="reward", bands=bands,
    label_region_mapping=label_region_mapping,
    region_to_anatomical_lobe=region_to_anatomical_lobe, difference="absolute"
)



plot_cluster_bubble_heatmap(
    summary,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    cluster_order=["Frontal", "Parietal", "Temporal", "Occipital",
                   "Insula", "Cingulate", "Limbic", "SCGM"],
    pat_state=pat_state_dict,
    sub_labels=sub_label_dict,
    vertical=False,
    show_sub_labels=True,
    abbrev_clusters=True,
    cbar_location="bottom",
    save_path=f"{plots_path}/fdr_source/pink_noise_reward/color/bubble/src_bubble_pinkcont_musiccont.pdf")


Aperiodic

In [66]:
eeg_fooof_pink_noise_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_pink_noise_reward_matched)


fdr_corrected_p_vals_pink_noise_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_pink_noise_reward_aperiodic, pat_state, n_channels=n_regions)


aperiodic_params = ["Exponent", "Offset"] 

save_path1 = f"{plots_path}/fdr_source/pink_noise_reward/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/pink_noise_reward/aperiodic/merged/"

parad1 = "Pink Noise Cont."
parad2 = "Music Cont."
view = "lateral"
pvals = True 


plot_fooof_fdr_results(fdr_corrected_p_vals_pink_noise_reward_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, 
                       cmap_color, view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_pinkcont_musiccont.pdf")  



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/aperiodic/sub09_Pink Noise Cont._Music Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/aperiodic/sub09_Pink Noise Cont._Music Cont._Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/aperiodic/sub10_Pink Noise Cont._Music Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink

In [67]:
save_path1 = f"{plots_path}/fdr_source/pink_noise_reward/aperiodic_color/"
save_path2 = f"{plots_path}/fdr_source/pink_noise_reward/aperiodic_color/merged/"

bands = ["Exponent", "Offset"]  # Aperiodic parameters
 
parad1 = "Pink Noise Cont."
parad2 = "Music Cont."
view = "lateral"




plot_fooof_relative_difference_results(
    eeg_fooof_pink_noise_reward_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="pink_noise", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2, out_filename="src_aperiodic_reldiff_pinkcont_musiccont.pdf", difference="absolute")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/aperiodic_color/sub09_pink_noise_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/aperiodic_color/sub09_pink_noise_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/aperiodic_color/sub10_pink_noise_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/aperiodic_color/sub10_pink_noise_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/aperiodic_color/sub12_pink_noise_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/

### 5. Rest Mid vs. Continuous Music

In [68]:
eeg_fooof_rest_mid_reward = {}

eeg_fooof_rest_mid_reward["sub07"] = {}
eeg_fooof_rest_mid_reward["sub07"]["rest_mid"] = eeg_fooof_all_subs["sub07"]["rest_mid"].copy()
eeg_fooof_rest_mid_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_mid_reward["sub09"] = {} 
eeg_fooof_rest_mid_reward["sub09"]["rest_mid"] = eeg_fooof_all_subs["sub09"]["rest_mid"].copy()
eeg_fooof_rest_mid_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_mid_reward["sub10"] = {} 
eeg_fooof_rest_mid_reward["sub10"]["rest_mid"] = eeg_fooof_all_subs["sub10"]["rest_mid"].copy()
eeg_fooof_rest_mid_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_rest_mid_reward["sub12"] = {} 
eeg_fooof_rest_mid_reward["sub12"]["rest_mid"] = eeg_fooof_all_subs["sub12"]["rest_mid"].copy()
eeg_fooof_rest_mid_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [69]:
eeg_fooof_rest_mid_reward_matched = get_matched_legths(eeg_fooof_rest_mid_reward)


Subject: sub07, Paradigms: ['rest_mid', 'reward']
Truncated array2 from 73 → 36 epochs (dropped last 37)
(90, 223, 36) (90, 223, 36)
Subject: sub09, Paradigms: ['rest_mid', 'reward']
Truncated array2 from 67 → 44 epochs (dropped last 23)
(90, 223, 44) (90, 223, 44)
Subject: sub10, Paradigms: ['rest_mid', 'reward']
Truncated array2 from 43 → 23 epochs (dropped last 20)
(90, 223, 23) (90, 223, 23)
Subject: sub12, Paradigms: ['rest_mid', 'reward']
Truncated array2 from 53 → 39 epochs (dropped last 14)
(90, 223, 39) (90, 223, 39)


In [70]:
sub_id = "sub10"

# freqs = psds_all[sub_id]["rest_init"][1]
# psds = psds_all[sub_id]["rest_init"][0]

freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/parads
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]   # only used for its region count (90) in get_fooof_bands

n_regions = 90

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}


_, rest_mid_reward_fooof_bands_pvalues_abs, fooof_rest_mid_reward = get_channel_wise_fooof(freqs, eeg_fooof_rest_mid_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_rest_mid_reward = get_fdr_p_values(rest_mid_reward_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)

sub07
Paradigm 0 rest_mid
Paradigm 1 reward
sub09
Paradigm 0 rest_mid
Paradigm 1 reward
sub10
Paradigm 0 rest_mid
Paradigm 1 reward
sub12
Paradigm 0 rest_mid
Paradigm 1 reward


In [71]:
save_path1 = f"{plots_path}/fdr_source/rest_mid_reward/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/rest_mid_reward/fdr_plots/merged/"

parad1 = "rest (Mid)"
parad2 = "Music Cont."
view = "lateral"
pvals = True



plot_fooof_fdr_results(fdr_corrected_p_vals_rest_mid_reward, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, 
                       pat_state_dict, save_path1, save_path2, out_filename="src_sig_restmid_musiccont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/fdr_plots/sub07_rest (Mid)_Music Cont._delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/fdr_plots/sub07_rest (Mid)_Music Cont._theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/fdr_plots/sub07_rest (Mid)_Music Cont._alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/fdr_plots/sub07_rest (Mid)_Music Cont._beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/fdr_plots/sub07_rest (Mid)_Music Cont._gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_n

In [72]:

parad1 = "REST (Mid)"
parad2 = "Music Cont."
view = "lateral"

_, eeg_fooof_bands_abs_rest_mid_reward = get_fooof_bands(eeg_fooof_rest_mid_reward_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_rest_mid_reward,
    bands, aal_img, label_region_mapping,
    condition1="rest_mid", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/rest_mid_reward/color/",
    save_dir2=f"{plots_path}/fdr_source/rest_mid_reward/color/merged/", out_filename="src_reldiff_restmid_musiccont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/color/sub07_rest_mid_reward_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/color/sub07_rest_mid_reward_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/color/sub07_rest_mid_reward_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/color/sub07_rest_mid_reward_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/color/sub07_rest_mid_reward_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/color/sub09_rest_mid_

Aperiodic

In [73]:
eeg_fooof_rest_mid_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_mid_reward_matched)


fdr_corrected_p_vals_rest_mid_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_mid_reward_aperiodic, 
                                                                                                                     pat_state, n_channels=n_regions)
aperiodic_params = ["Exponent", "Offset"] 

save_path1 = f"{plots_path}/fdr_source/rest_mid_reward/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/rest_mid_reward/aperiodic/merged/"

parad1 = "Rest (Mid)"
parad2 = "Music Cont."
view = "lateral"
pvals = True 


plot_fooof_fdr_results(fdr_corrected_p_vals_rest_mid_reward_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, 
                       cmap_color, view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_restmid_reward_musiccont.pdf")  



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/aperiodic/sub07_Rest (Mid)_Music Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/aperiodic/sub07_Rest (Mid)_Music Cont._Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/aperiodic/sub09_Rest (Mid)_Music Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_stu

In [74]:
save_path1 = f"{plots_path}/fdr_source/rest_mid_reward/aperiodic_color/"
save_path2 = f"{plots_path}/fdr_source/rest_mid_reward/aperiodic_color/merged/"

bands = ["Exponent", "Offset"]  # Aperiodic parameters
 
parad1 = "Rest (Mid)"
parad2 = "Music Cont."
view = "lateral"

plot_fooof_relative_difference_results(
    eeg_fooof_rest_mid_reward_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="rest_mid", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2, out_filename="src_aperiodic_reldiff_restmid_reward_musiccont.pdf", difference="absolute")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/aperiodic_color/sub07_rest_mid_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/aperiodic_color/sub07_rest_mid_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/aperiodic_color/sub09_rest_mid_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/aperiodic_color/sub09_rest_mid_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/aperiodic_color/sub10_rest_mid_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vecto

### 6. Rest End. vs. Music Cont.

In [75]:
eeg_fooof_rest_end_reward = {}

eeg_fooof_rest_end_reward["sub07"] = {}
eeg_fooof_rest_end_reward["sub07"]["rest_end"] = eeg_fooof_all_subs["sub07"]["rest_end"].copy()
eeg_fooof_rest_end_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_end_reward["sub09"] = {} 
eeg_fooof_rest_end_reward["sub09"]["rest_end"] = eeg_fooof_all_subs["sub09"]["rest_end"].copy()
eeg_fooof_rest_end_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_end_reward["sub12"] = {} 
eeg_fooof_rest_end_reward["sub12"]["rest_end"] = eeg_fooof_all_subs["sub12"]["rest_end"].copy()
eeg_fooof_rest_end_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [76]:
eeg_fooof_rest_end_reward_matched = get_matched_legths(eeg_fooof_rest_end_reward)


Subject: sub07, Paradigms: ['rest_end', 'reward']
Truncated array2 from 73 → 37 epochs (dropped last 36)
(90, 223, 37) (90, 223, 37)
Subject: sub09, Paradigms: ['rest_end', 'reward']
Truncated array2 from 67 → 44 epochs (dropped last 23)
(90, 223, 44) (90, 223, 44)
Subject: sub12, Paradigms: ['rest_end', 'reward']
Truncated array2 from 53 → 36 epochs (dropped last 17)
(90, 223, 36) (90, 223, 36)


In [77]:
sub_id = "sub10"

# freqs = psds_all[sub_id]["rest_init"][1]
# psds = psds_all[sub_id]["rest_init"][0]

freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/parads
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]   # only used for its region count (90) in get_fooof_bands

n_regions = 90

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}


_, rest_end_reward_fooof_bands_pvalues_abs, fooof_rest_end_reward = get_channel_wise_fooof(freqs, eeg_fooof_rest_end_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_rest_end_reward = get_fdr_p_values(rest_end_reward_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)

sub07
Paradigm 0 rest_end
Paradigm 1 reward
sub09
Paradigm 0 rest_end
Paradigm 1 reward
sub12
Paradigm 0 rest_end
Paradigm 1 reward


In [78]:
save_path1 = f"{plots_path}/fdr_source/rest_end_reward/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/rest_end_reward/fdr_plots/merged/"

parad1 = "rest (End)"
parad2 = "Music Cont."
view = "lateral"
pvals = True



plot_fooof_fdr_results(fdr_corrected_p_vals_rest_end_reward, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, 
                       pat_state_dict, save_path1, save_path2, out_filename="src_sig_restend_musiccont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/fdr_plots/sub07_rest (End)_Music Cont._delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/fdr_plots/sub07_rest (End)_Music Cont._theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/fdr_plots/sub07_rest (End)_Music Cont._alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/fdr_plots/sub07_rest (End)_Music Cont._beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/fdr_plots/sub07_rest (End)_Music Cont._gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_n

In [79]:

parad1 = "REST (End)"
parad2 = "Music Cont."
view = "lateral"

_, eeg_fooof_bands_abs_rest_end_reward = get_fooof_bands(eeg_fooof_rest_end_reward_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_rest_end_reward,
    bands, aal_img, label_region_mapping,
    condition1="rest_end", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/rest_end_reward/color/",
    save_dir2=f"{plots_path}/fdr_source/rest_end_reward/color/merged/", out_filename="src_reldiff_restend_musiccont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/color/sub07_rest_end_reward_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/color/sub07_rest_end_reward_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/color/sub07_rest_end_reward_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/color/sub07_rest_end_reward_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/color/sub07_rest_end_reward_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/color/sub09_rest_end_

Aperiodic

In [80]:
eeg_fooof_rest_end_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_end_reward_matched)


fdr_corrected_p_vals_rest_end_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_end_reward_aperiodic, 
                                                                                                                     pat_state, n_channels=n_regions)
aperiodic_params = ["Exponent", "Offset"] 

save_path1 = f"{plots_path}/fdr_source/rest_end_reward/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/rest_end_reward/aperiodic/merged/"

parad1 = "Rest (End)"
parad2 = "Music Cont."
view = "lateral"
pvals = True 


plot_fooof_fdr_results(fdr_corrected_p_vals_rest_end_reward_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, 
                       cmap_color, view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_restend_reward_musiccont.pdf")  



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic/sub07_Rest (End)_Music Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic/sub07_Rest (End)_Music Cont._Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic/sub09_Rest (End)_Music Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic/sub09_Rest (En

In [81]:
save_path1 = f"{plots_path}/fdr_source/rest_end_reward/aperiodic_color/"
save_path2 = f"{plots_path}/fdr_source/rest_end_reward/aperiodic_color/merged/"

bands = ["Exponent", "Offset"]  # Aperiodic parameters
 
parad1 = "Rest (End)"
parad2 = "Music Cont."
view = "lateral"

plot_fooof_relative_difference_results(
    eeg_fooof_rest_end_reward_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="rest_end", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2, out_filename="src_aperiodic_reldiff_restend_reward_musiccont.pdf", difference="absolute")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic_color/sub07_rest_end_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic_color/sub07_rest_end_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic_color/sub09_rest_end_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic_color/sub09_rest_end_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/aperiodic_color/sub12_rest_end_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vecto

### 7. Rest vs. Pink Noise Continuous

In [ ]:
# eeg_fooof_rest_pink_cont = {}

# eeg_fooof_rest_pink_cont["sub09"] = {}
# eeg_fooof_rest_pink_cont["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
# eeg_fooof_rest_pink_cont["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()

# eeg_fooof_rest_pink_cont["sub10"] = {} 
# eeg_fooof_rest_pink_cont["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
# eeg_fooof_rest_pink_cont["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()

# eeg_fooof_rest_pink_cont["sub12"] = {} 
# eeg_fooof_rest_pink_cont["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
# eeg_fooof_rest_pink_cont["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()

In [ ]:
# eeg_fooof_rest_pink_cont_matched = get_matched_legths(eeg_fooof_rest_pink_cont)


Subject: sub09, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 88 → 51 epochs (dropped last 37)
(90, 223, 51) (90, 223, 51)
Subject: sub10, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 48 → 36 epochs (dropped last 12)
(90, 223, 36) (90, 223, 36)
Subject: sub12, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 70 → 58 epochs (dropped last 12)
(90, 223, 58) (90, 223, 58)


In [ ]:
# sub_id = "sub10"

# # freqs = psds_all[sub_id]["rest_init"][1]
# # psds = psds_all[sub_id]["rest_init"][0]

# freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/parads
# psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]   # only used for its region count (90) in get_fooof_bands

# n_regions = 90

# bands = {
#     "delta": (0.5, 4),
#     "theta": (4, 8),
#     "alpha": (8, 12),
#     "beta": (12, 30),
#     "gamma low": (30, 45),
# }


# _, rest_pink_cont_fooof_bands_pvalues_abs, fooof_rest_pink_cont = get_channel_wise_fooof(freqs, eeg_fooof_rest_pink_cont_matched, bands, channels_n=n_regions)
# fdr_corrected_p_vals_rest_pink_cont = get_fdr_p_values(rest_pink_cont_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)

sub09
Paradigm 0 rest_init
Paradigm 1 pink_noise
sub10
Paradigm 0 rest_init
Paradigm 1 pink_noise
sub12
Paradigm 0 rest_init
Paradigm 1 pink_noise


In [ ]:
# save_path1 = f"{plots_path}/fdr_source/rest_pink_cont/fdr_plots/"
# save_path2 = f"{plots_path}/fdr_source/rest_pink_cont/fdr_plots/merged/"

# parad1 = "rest (End)"
# parad2 = "Music Cont."
# view = "lateral"
# pvals = True



# plot_fooof_fdr_results(fdr_corrected_p_vals_rest_pink_cont, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, 
#                        pat_state_dict, save_path1, save_path2, out_filename="src_sig_restinit_pinkcont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/fdr_plots/sub09_rest (End)_Music Cont._delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/fdr_plots/sub09_rest (End)_Music Cont._theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/fdr_plots/sub09_rest (End)_Music Cont._alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/fdr_plots/sub09_rest (End)_Music Cont._beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/fdr_plots/sub09_rest (End)_Music Cont._gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt

In [ ]:
# _, eeg_fooof_bands_abs_rest_pink_cont = get_fooof_bands(eeg_fooof_rest_pink_cont_matched, freqs, psds, bands)


# plot_fooof_relative_difference_results(
#     eeg_fooof_bands_abs_rest_pink_cont,
#     bands, aal_img, label_region_mapping,
#     condition1="rest_init", condition2="pink_noise",
#     cmap_color=soft_div, view="lateral",
#     pat_state_list=pat_state_dict,
#     save_dir1=f"{plots_path}/fdr_source/rest_pink_cont/color/",
#     save_dir2=f"{plots_path}/fdr_source/rest_pink_cont/color/merged/", out_filename="src_reldiff_restinit_pinkcont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/color/sub09_rest_init_pink_noise_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/color/sub09_rest_init_pink_noise_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/color/sub09_rest_init_pink_noise_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/color/sub09_rest_init_pink_noise_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/color/sub09_rest_init_pink_noise_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/co

Aperiodic

In [ ]:
# eeg_fooof_rest_pink_cont_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_pink_cont_matched)


# fdr_corrected_p_vals_rest_pink_cont_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_pink_cont_aperiodic, 
#                                                                                                                      pat_state, n_channels=n_regions)
# aperiodic_params = ["Exponent", "Offset"] 

# save_path1 = f"{plots_path}/fdr_source/rest_pink_cont/aperiodic/"
# save_path2 = f"{plots_path}/fdr_source/rest_pink_cont/aperiodic/merged/"

# parad1 = "Rest Init"
# parad2 = "Pink Cont."
# view = "lateral"
# pvals = True 


# plot_fooof_fdr_results(fdr_corrected_p_vals_rest_pink_cont_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, 
#                        cmap_color, view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_restinit_pinkcont.pdf")  



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aperiodic/sub09_Rest Init_Pink Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aperiodic/sub09_Rest Init_Pink Cont._Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aperiodic/sub10_Rest Init_Pink Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aper

In [ ]:
# save_path1 = f"{plots_path}/fdr_source/rest_pink_cont/aperiodic_color/"
# save_path2 = f"{plots_path}/fdr_source/rest_pink_cont/aperiodic_color/merged/"

# bands = ["Exponent", "Offset"]  # Aperiodic parameters
 

# plot_fooof_relative_difference_results(
#     eeg_fooof_rest_pink_cont_aperiodic,
#     bands, aal_img, label_region_mapping,
#     condition1="rest_init", condition2="pink_noise",
#     cmap_color=soft_div, view="lateral",
#     pat_state_list=pat_state_dict,
#     save_dir1=save_path1,
#     save_dir2=save_path2, out_filename="src_aperiodic_reldiff_restinit_pinkcont.pdf", difference="absolute")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aperiodic_color/sub09_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aperiodic_color/sub09_rest_init_pink_noise_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aperiodic_color/sub10_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aperiodic_color/sub10_rest_init_pink_noise_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/aperiodic_color/sub12_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/

### Dual Plots

### Periodic

Rest Init vs. Reward

In [82]:
folder_a = plots_path+ "/fdr_source/rest_reward/color/"
folder_b = plots_path + "/fdr_source/rest_reward/fdr_plots/"
outpur_folder = plots_path + "/fdr_source/rest_reward/combined_dual_grid/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_rest_musiccont.pdf",
    within_gap_y_in=0.0,
    transpose=True,
    show_sub_labels=1,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)
# build_vector_grid_pdf_dual_folder(
#     image_dir_a=folder_a,
#     image_dir_b=folder_b,
#     bands=["delta", "theta", "alpha", "beta", "gamma low"],
#     pat_state=pat_state_dict,
#     save_dir=outpur_folder,
#     out_filename="src_dual_rest_musiccont.pdf",
#     within_gap_y_in=0.0,
#     transpose=True,
#     show_sub_labels=True,
#     sub_label_position="bottom",
#     # row_label_width_in / row_label_fontsize: width now unused; fontsize styles the bottom labels
#     col_label_fontsize=26,
#     col_label_height_in=0.75,
# )

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/combined_dual_grid/src_dual_rest_musiccont.pdf


Rest Mid vs. Reward

In [83]:
folder_a = plots_path+ "/fdr_source/rest_mid_reward/color/"
folder_b = plots_path + "/fdr_source/rest_mid_reward/fdr_plots/"
outpur_folder = plots_path + "/fdr_source/rest_mid_reward/combined_dual_grid/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_rest_mid_musiccont.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/combined_dual_grid/src_dual_rest_mid_musiccont.pdf


Rest vs. Pink Noise Cont.

In [ ]:
# folder_a = plots_path+ "/fdr_source/rest_pink_cont/color/"
# folder_b = plots_path + "/fdr_source/rest_pink_cont/fdr_plots/"
# outpur_folder = plots_path + "/fdr_source/rest_pink_cont/combined_dual_grid/"

# build_vector_grid_pdf_dual_folder(
#     image_dir_a=folder_a,
#     image_dir_b=folder_b,
#     bands=["delta", "theta", "alpha", "beta", "gamma low"],
#     pat_state=pat_state_dict,
#     save_dir=outpur_folder,
#     out_filename="src_dual_rest_pink_cont.pdf",
#     within_gap_y_in=0.0,
#     transpose=False,
#     show_sub_labels=True,
#     row_label_width_in=1.6,
#     row_label_fontsize=20,
#     col_label_fontsize=20,
#     col_label_height_in=0.75,
# )

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/combined_dual_grid/src_dual_rest_pink_cont.pdf


Rest End vs. Reward

In [84]:
folder_a = plots_path+ "/fdr_source/rest_end_reward/color/"
folder_b = plots_path + "/fdr_source/rest_end_reward/fdr_plots/"
outpur_folder = plots_path + "/fdr_source/rest_end_reward/combined_dual_grid/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_rest_end_musiccont.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/combined_dual_grid/src_dual_rest_end_musiccont.pdf


Rest vs. Pink Noise

In [114]:
folder_a = plots_path+ "/fdr_source/rest_pink_noise/color/"
folder_b = plots_path + "/fdr_source/rest_pink_noise/fdr_plots/"
outpur_folder = plots_path + "/fdr_source/rest_pink_noise/combined_dual_grid/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_rest_pink_noise.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/combined_dual_grid/src_dual_rest_pink_noise.pdf


Pink Noise vs. Reward

In [86]:
folder_a = plots_path + "/fdr_source/pink_noise_reward/color/"
folder_b = plots_path + "/fdr_source/pink_noise_reward/fdr_plots/"
outpur_folder = plots_path + "/fdr_source/pink_noise_reward/combined_dual_grid/"


build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_pinkcont_musiccont.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)


Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/combined_dual_grid/src_dual_pinkcont_musiccont.pdf


Music Segs vs. Pink Segs

In [87]:
folder_a = plots_path + "/fdr_source/music_pink_segs/color/"
folder_b = plots_path + "/fdr_source/music_pink_segs/fdr_plots/"
outpur_folder = plots_path + "/fdr_source/music_pink_segs/combined_dual_grid/"


build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_musicsegs_pinksegs.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)


Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/combined_dual_grid/src_dual_musicsegs_pinksegs.pdf


### Aperiodic

Rest Init vs. Reward

In [43]:
folder_a = plots_path+ "/fdr_source/rest_reward/aperiodic_color/"
folder_b = plots_path + "/fdr_source/rest_reward/aperiodic/"
outpur_folder = plots_path + "/fdr_source/rest_reward/combined_dual_grid_aperiodic/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["Offset", "Exponent"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_aperiodic_dual_rest_musiccont.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=0.35,
    row_label_fontsize=24,
    col_label_fontsize=23,
    col_label_height_in=0.75,
    show_band_labels=True
)

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_reward/combined_dual_grid_aperiodic/src_aperiodic_dual_rest_musiccont.pdf


Rest Init vs Pink Noise Cont.

In [ ]:
# folder_a = plots_path+ "/fdr_source/rest_pink_cont/aperiodic_color/"
# folder_b = plots_path + "/fdr_source/rest_pink_cont/aperiodic/"
# outpur_folder = plots_path + "/fdr_source/rest_pink_cont/combined_dual_grid_aperiodic/"

# build_vector_grid_pdf_dual_folder(
#     image_dir_a=folder_a,
#     image_dir_b=folder_b,
#     bands=["Offset", "Exponent"],
#     pat_state=pat_state_dict,
#     save_dir=outpur_folder,
#     out_filename="src_aperiodic_dual_rest_pink_cont.pdf",
#     within_gap_y_in=0.0,
#     transpose=False,
#     show_sub_labels=True,
#     row_label_width_in=1.6,
#     row_label_fontsize=20,
#     col_label_fontsize=22,
#     col_label_height_in=0.75,
#     show_band_labels=False
# )

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_cont/combined_dual_grid_aperiodic/src_aperiodic_dual_rest_pink_cont.pdf


Rest Mid vs. Reward

In [89]:
folder_a = plots_path+ "/fdr_source/rest_mid_reward/aperiodic_color/"
folder_b = plots_path + "/fdr_source/rest_mid_reward/aperiodic/"
outpur_folder = plots_path + "/fdr_source/rest_mid_reward/combined_dual_grid_aperiodic/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["Offset", "Exponent"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_aperiodic_dual_rest_mid_musiccont.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_mid_reward/combined_dual_grid_aperiodic/src_aperiodic_dual_rest_mid_musiccont.pdf


Rest End vs. Reward

In [90]:
folder_a = plots_path+ "/fdr_source/rest_end_reward/aperiodic_color/"
folder_b = plots_path + "/fdr_source/rest_end_reward/aperiodic/"
outpur_folder = plots_path + "/fdr_source/rest_end_reward/combined_dual_grid_aperiodic/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["Offset", "Exponent"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_aperiodic_dual_rest_end_musiccont.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_end_reward/combined_dual_grid_aperiodic/src_aperiodic_dual_rest_end_musiccont.pdf


Rest vs. Pink Noise

In [91]:
folder_a = plots_path+ "/fdr_source/rest_pink_noise/aperiodic_color/"
folder_b = plots_path + "/fdr_source/rest_pink_noise/aperiodic/"
outpur_folder = plots_path + "/fdr_source/rest_pink_noise/combined_dual_grid_aperiodic/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["Offset", "Exponent"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_aperiodic_dual_rest_pink_noise.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)

Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/rest_pink_noise/combined_dual_grid_aperiodic/src_aperiodic_dual_rest_pink_noise.pdf


Pink Noise vs. Reward

In [92]:
folder_a = plots_path + "/fdr_source/pink_noise_reward/aperiodic_color/"
folder_b = plots_path + "/fdr_source/pink_noise_reward/aperiodic/"
outpur_folder = plots_path + "/fdr_source/pink_noise_reward/combined_dual_grid_aperiodic/"


build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["Offset", "Exponent"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_aperiodic_dual_pinkcont_musiccont.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)


Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/pink_noise_reward/combined_dual_grid_aperiodic/src_aperiodic_dual_pinkcont_musiccont.pdf


Pink Segs vs. Music Segs

In [93]:
folder_a = plots_path + "/fdr_source/music_pink_segs/aperiodic_color/"
folder_b = plots_path + "/fdr_source/music_pink_segs/aperiodic/"
outpur_folder = plots_path + "/fdr_source/music_pink_segs/combined_dual_grid_aperiodic/"


build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["Offset", "Exponent"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_aperiodic_dual_musicsegs_pinksegs.pdf",
    within_gap_y_in=0.0,
    transpose=False,
    show_sub_labels=True,
    row_label_width_in=1.6,
    row_label_fontsize=20,
    col_label_fontsize=20,
    col_label_height_in=0.75,
)


Saved (raster panels @ 200 dpi, vector labels) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt//fdr_source/music_pink_segs/combined_dual_grid_aperiodic/src_aperiodic_dual_musicsegs_pinksegs.pdf


In [97]:
# import fitz
# for path in [
#     "music_segs_pink_segs_abs_fdr.pdf",
#     "music_segs_pink_segs_abs.pdf",
#     "music_segs_pink_segs_abs_aperiodic_fdr.pdf",
#     "music_segs_pink_segs_abs_aperiodic.pdf",
# ]:
#     doc = fitz.open(path)
#     r = doc[0].rect
#     print(f"{path}: {r.width:.1f} x {r.height:.1f} pt, aspect ratio = {r.width/r.height:.3f}")
#     doc.close()